# Blog 10 — Unity Catalog

## Databricks Governance, Namespace, Tables, Volumes and Access Control

> **Goal:** understand Unity Catalog conceptually and practically without mixing it with Auto Loader, Structured Streaming, Jobs or Pipelines.

This notebook covers:

1. What Unity Catalog is
2. Why it exists
3. Metastore → Catalog → Schema → Object hierarchy
4. Catalogs
5. Schemas
6. Tables
7. Managed vs External Tables
8. Volumes
9. Managed vs External Volumes
10. Views
11. Three-level namespace
12. `USE CATALOG` / `USE SCHEMA`
13. Permissions and privileges
14. `GRANT` / `REVOKE`
15. Ownership
16. Users / Groups / Service Principals
17. Data Governance
18. Lineage
19. Unity Catalog + Delta Lake
20. Unity Catalog + Medallion Architecture

# 1. What is Unity Catalog?

**Unity Catalog (UC)** is Databricks' centralized governance and metadata layer for data and AI assets.

In simple terms:

> **Unity Catalog organizes data assets, controls access to them, and provides a consistent namespace and governance model across Databricks.**

Without a governance layer:

```text
Random storage locations
        ↓
Random tables
        ↓
Different teams
        ↓
Unclear ownership
        ↓
Unclear permissions
        ↓
Difficult governance
```

With Unity Catalog:

```text
                Unity Catalog
                     │
          ┌──────────┴──────────┐
          │                     │
       Catalog               Catalog
          │                     │
       Schema                Schema
          │                     │
    ┌─────┼─────┐        ┌────┼─────┐
  Tables Views Volumes   Tables Views Volumes
```

# 2. Why Unity Catalog exists

Imagine an organization has:

```text
Sales
Finance
Marketing
Operations
Data Science
```

and thousands of datasets.

The organization needs to answer:

- Who owns this table?
- Who can read it?
- Who can modify it?
- Which schema does it belong to?
- Where is the data stored?
- Which team owns it?
- Which downstream tables depend on it?
- Can analysts access sensitive data?
- Can one team accidentally modify another team's data?

Unity Catalog provides the governance framework for these questions.

### Core responsibilities

```text
Organization
+
Namespace
+
Access Control
+
Ownership
+
Discovery
+
Lineage
+
Governance
```

# 3. Unity Catalog hierarchy

This is one of the most important concepts in Unity Catalog.

Conceptually:

```text
Metastore
    │
    ├── Catalog
    │      │
    │      ├── Schema
    │      │      │
    │      │      ├── Table
    │      │      ├── View
    │      │      └── Volume
    │      │
    │      └── Schema
    │
    └── Catalog
```

For everyday Data Engineering work, remember:

```text
Catalog
   ↓
Schema
   ↓
Object
```

For example:

```text
workspace.medallion_project.orders
```

where:

```text
workspace         = Catalog
medallion_project = Schema
orders            = Table
```

# 4. Metastore

The **metastore** is the top-level governance boundary for Unity Catalog metadata.

Conceptually:

```text
Metastore
   │
   ├── Catalog A
   ├── Catalog B
   └── Catalog C
```

The metastore is important to understand conceptually, but Data Engineers usually work more frequently with:

```text
Catalog
Schema
Table
View
Volume
Permissions
```

# 5. Catalogs

A **catalog** is a top-level organizational container underneath the metastore.

Examples could include:

```text
production
development
analytics
finance
marketing
```

For example:

```text
production.sales.orders
production.sales.customers
development.sales.orders
```

A catalog can be used as a major organizational boundary based on an organization's governance and architecture strategy.

### Mental model

> **Catalog = major organizational / governance boundary**

# 6. Schemas

A **schema** sits inside a catalog.

For example:

```text
production
    │
    ├── sales
    ├── finance
    └── marketing
```

Here:

```text
production = catalog
sales      = schema
```

The schema contains objects such as:

```text
tables
views
volumes
```

Example:

```text
production.sales.orders
production.sales.customers
production.sales.order_items
```

# 7. Catalog vs Schema vs Table

| Level | Example | Purpose |
|---|---|---|
| Catalog | `production` | Major organizational boundary |
| Schema | `sales` | Collection of related objects |
| Table | `orders` | Structured data |

Therefore:

```text
production.sales.orders
```

means:

```text
Catalog.Schema.Table
```

# 8. Tables

A table contains structured data.

Example:

```text
production.sales.orders
```

could contain:

| order_id | customer_id | amount | status |
|---:|---:|---:|---|
| 1001 | 501 | 1250.50 | COMPLETED |
| 1002 | 502 | 850.00 | PENDING |

In Databricks, these will commonly be Delta tables.

Example:

```sql
SELECT *
FROM production.sales.orders;
```

In [ ]:
# Basic table inspection

# Replace these names with an object that exists in your environment.

# Example:
# display(
#     spark.sql(
#         "SELECT * FROM workspace.medallion_project.orders LIMIT 10"
#     )
# )


# 9. Managed Tables

A **managed table** is a table where Databricks manages the table's underlying storage lifecycle according to the catalog configuration.

Example:

```sql
CREATE TABLE workspace.uc_demo.orders (
    order_id BIGINT,
    customer_id BIGINT,
    amount DOUBLE
);
```

You do not specify a physical storage location in the basic example.

### Mental model

```text
CREATE TABLE
     ↓
Unity Catalog registers table
     ↓
Databricks manages storage lifecycle
```

# 10. External Tables

An **external table** registers data that already exists at an external storage location.

Conceptually:

```text
Cloud Storage
      │
      ▼
Existing Delta data
      │
      ▼
Unity Catalog external table
```

Example pattern:

```sql
CREATE TABLE workspace.uc_demo.orders
USING DELTA
LOCATION '...';
```

### Important distinction

> **Managed vs external describes the relationship between the table and its underlying storage lifecycle.**

Both can be governed through Unity Catalog.

# 11. Managed vs External Tables

| | Managed | External |
|---|---|---|
| Data location | Managed by platform configuration | Explicit external location |
| Table registered in UC | Yes | Yes |
| UC governance | Yes | Yes |
| Useful for | New managed lakehouse data | Existing/external storage |

The key Data Engineer question is:

> **Who controls the underlying data lifecycle and where is the data located?**

# 12. Volumes

A **Volume** is a Unity Catalog-governed file storage object.

Tables are primarily for structured tabular data.

Volumes are useful for files such as:

```text
JSON
CSV
PDF
images
documents
raw files
ML artifacts
configuration files
```

Conceptually:

```text
Catalog
   ↓
Schema
   ↓
Volume
   ↓
Files
```

Example path:

```text
/Volumes/workspace/my_schema/project_volume/source/
```

# 13. Why Volumes matter

Volumes are especially useful for file-oriented workloads.

For example:

```text
workspace
    ↓
blog10
    ↓
project_volume
    ├── source/
    ├── schemas/
    └── files/
```

This provides a governed location for files inside the Unity Catalog model.

This is also why the later Auto Loader work can use Unity Catalog Volume paths.

In [ ]:
# Example Volume path

CATALOG = "workspace"
SCHEMA = "uc_demo"
VOLUME = "project_volume"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print(VOLUME_PATH)

# Example:
# /Volumes/workspace/uc_demo/project_volume


# 14. Managed vs External Volumes

Conceptually:

### Managed Volume

Unity Catalog manages the storage lifecycle.

### External Volume

The volume points to an external storage location governed through Unity Catalog.

The important mental model is:

```text
Volume
   ↓
File-oriented storage
   ↓
Unity Catalog governance
```

Do not confuse a Volume with a Table:

```text
Table  → structured/tabular data
Volume → files
```

# 15. Views

A **view** is a logical representation of data based on a SQL query.

Example:

```sql
CREATE VIEW workspace.sales.completed_orders AS
SELECT *
FROM workspace.sales.orders
WHERE status = 'COMPLETED';
```

Conceptually:

```text
orders table
     ↓
   query
     ↓
completed_orders view
```

Views are useful for:

- reusable queries
- abstraction
- controlled data presentation
- simplifying analytics
- exposing only appropriate records or columns

# 16. Three-level namespace

This is one of the most important Unity Catalog concepts.

The standard namespace is:

```text
catalog.schema.object
```

Example:

```text
workspace.sales.orders
```

means:

```text
workspace → catalog
sales     → schema
orders    → table
```

### Important rule

For Unity Catalog, do not invent extra namespace levels.

The normal structure is:

```text
CATALOG
   ↓
SCHEMA
   ↓
OBJECT
```

In [ ]:
# Three-level namespace examples

examples = [
    "workspace.sales.orders",
    "workspace.finance.transactions",
    "workspace.analytics.customer_summary"
]

for name in examples:
    print(name)


# 17. `USE CATALOG`

You can set the current catalog:

```sql
USE CATALOG workspace;
```

After this, commands operate in the selected catalog context.

You can inspect schemas with:

```sql
SHOW SCHEMAS;
```

In [ ]:
# SQL context example

spark.sql("USE CATALOG workspace")

print("Current catalog context set to: workspace")

display(
    spark.sql("SHOW SCHEMAS")
)


# 18. `USE SCHEMA`

After selecting the catalog, select a schema:

```sql
USE SCHEMA medallion_project;
```

Your current context becomes:

```text
workspace.medallion_project
```

You can then reference an object more simply:

```sql
SELECT *
FROM orders;
```

instead of:

```sql
SELECT *
FROM workspace.medallion_project.orders;
```

In [ ]:
# Example schema context

# Run only if the schema exists in your workspace.
#
# spark.sql("USE CATALOG workspace")
# spark.sql("USE SCHEMA medallion_project")
#
# display(spark.sql("SHOW TABLES"))


# 19. Fully qualified names

Even when a catalog and schema are selected, fully qualified names are often useful in production SQL.

Example:

```sql
SELECT *
FROM workspace.medallion_project.orders;
```

Advantages:

- explicit dependency
- easier code review
- easier debugging
- avoids ambiguity
- makes cross-schema references clear

# 20. Permissions and privileges

Unity Catalog is not only about organization.

It controls access.

Conceptually:

```text
User / Group / Service Principal
              │
              ▼
           Privilege
              │
              ▼
Catalog / Schema / Table / View / Volume
```

Examples of privileges include:

```text
USE CATALOG
USE SCHEMA
SELECT
MODIFY
CREATE TABLE
CREATE VIEW
READ VOLUME
WRITE VOLUME
```

The available privileges depend on the object type.

# 21. `GRANT`

Permissions can be granted to identities.

Example:

```sql
GRANT SELECT
ON TABLE workspace.sales.orders
TO `analyst_group`;
```

Conceptually:

```text
analyst_group
      ↓
   SELECT
      ↓
workspace.sales.orders
```

The practical principle is:

> Give an identity only the access it needs to perform its job.

# 22. `REVOKE`

Privileges can also be removed.

Example:

```sql
REVOKE SELECT
ON TABLE workspace.sales.orders
FROM `analyst_group`;
```

This removes that specific grant.

In production, access should be managed deliberately rather than giving broad permissions to everyone.

# 23. Permissions across the hierarchy

Permissions can be applied at different levels:

```text
Catalog
   ↓
Schema
   ↓
Table / View / Volume
```

This allows organizations to design access at an appropriate granularity.

For example:

```text
Data Engineering group
    ↓
broader schema privileges

Analytics group
    ↓
SELECT access

Application identity
    ↓
specific required objects
```

The exact privilege model and inheritance behavior should always be checked against the organization's Databricks configuration and current documentation.

# 24. Users, Groups and Service Principals

### User

A human identity.

```text
Data Engineer
Analyst
Manager
```

### Group

A collection of users.

```text
data_engineers
analysts
finance_team
```

### Service Principal

A machine/application identity.

For example:

```text
production_pipeline
```

could use a service principal rather than a human identity.

### Production principle

Prefer scalable identity management through groups and service principals rather than designing permissions around individual users whenever possible.

# 25. Ownership

Unity Catalog objects have owners.

Conceptually:

```text
Table
  ↓
Owner
  ↓
Responsible identity
```

Ownership matters because someone needs responsibility for:

- managing the object
- managing access
- maintaining the data asset
- handling governance responsibilities

Ownership is part of data governance, not simply administration.

# 26. Data governance

Unity Catalog brings multiple governance capabilities together:

```text
Data organization
        +
Namespace
        +
Access control
        +
Ownership
        +
Discovery
        +
Lineage
        +
Auditing / governance
```

Therefore:

> **Unity Catalog is a governance layer, not simply a table registry.**

# 27. Lineage

Lineage answers:

> **Where did this data come from, and where does it go?**

Example:

```text
Raw Orders
     ↓
Bronze Orders
     ↓
Silver Orders
     ↓
Gold Revenue
     ↓
Dashboard
```

If a Gold metric looks wrong, lineage can help trace its upstream dependencies.

This becomes increasingly important as the number of pipelines and tables grows.

# 28. Unity Catalog + Delta Lake

Unity Catalog and Delta Lake have different responsibilities.

```text
Unity Catalog
      ↓
Governance
Namespace
Permissions
Ownership
Discovery
Lineage

Delta Lake
      ↓
Transactional table storage
ACID transactions
Schema enforcement
Time travel
MERGE
Optimization
```

Together:

```text
Unity Catalog
      +
Delta Lake
      ↓
Governed lakehouse tables
```

This distinction is extremely important.

# 29. Unity Catalog + Medallion Architecture

Your existing Medallion Architecture can sit inside Unity Catalog.

Example:

```text
workspace
└── medallion_project
    ├── bronze_orders
    ├── silver_orders
    └── gold_sales
```

Conceptually:

```text
                 UNITY CATALOG
                       │
                  workspace
                       │
              medallion_project
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
     Bronze          Silver          Gold
        │              │              │
        └──────────────┴──────────────┘
                       │
                    Delta
```

Unity Catalog governs the assets.

Delta provides the transactional table layer.

The Medallion architecture provides the data-processing organization.

# 30. What Unity Catalog does NOT replace

### Unity Catalog does not replace Delta

```text
Delta → table/storage format
UC    → governance/catalog
```

### Unity Catalog does not replace Jobs

```text
Jobs → orchestration
UC   → governance
```

### Unity Catalog does not replace Auto Loader

```text
Auto Loader → ingestion
UC          → governance
```

### Unity Catalog does not replace Data Quality

```text
Data Quality → correctness
UC           → governance
```

Think of them as different layers of the platform.

# 31. Complete architecture mental model

```text
                         DATABRICKS
                             │
                    ┌────────┴────────┐
                    │                 │
              UNITY CATALOG        COMPUTE
                    │
             Governance layer
                    │
              ┌─────┴─────┐
              │           │
           Catalog      Catalog
              │
           Schema
              │
      ┌───────┼────────┐
      ▼       ▼        ▼
    Tables   Views   Volumes
      │
    Delta
      │
 ┌────┼─────┐
 ▼    ▼     ▼
Bronze Silver Gold
```

Around this:

```text
Auto Loader  → ingestion
Jobs         → orchestration
Delta        → storage
Unity Catalog→ governance
Data Quality → correctness
```

# 32. What you should master after Blog 10

## Must know deeply

- Metastore
- Catalog
- Schema
- Tables
- Managed vs External Tables
- Volumes
- Three-level namespace
- `USE CATALOG`
- `USE SCHEMA`
- Permissions
- Privileges
- `GRANT`
- `REVOKE`
- Ownership
- Unity Catalog + Delta

## Understand well

- Managed vs External Volumes
- Views
- Users vs Groups vs Service Principals
- Governance
- Lineage

## Conceptual knowledge is enough for now

- Deep metastore administration
- Complex enterprise identity architecture
- Advanced governance implementation
- Detailed lineage internals

Those can be learned when a specific role requires them.

# 33. Interview-level mental model

If asked:

### What is Unity Catalog?

> Unity Catalog is Databricks' centralized governance and metadata layer that provides a hierarchical namespace, access control, ownership, discovery and lineage for data and AI assets.

### What is the hierarchy?

```text
Metastore
   ↓
Catalog
   ↓
Schema
   ↓
Table / View / Volume
```

### What is the three-level namespace?

```text
catalog.schema.object
```

### Managed vs External Table?

> Managed tables have their storage lifecycle managed by Databricks according to the catalog configuration, while external tables register data stored at an external location.

### Table vs Volume?

> Tables represent structured/tabular data; Volumes provide governed file-oriented storage.

# 34. Blog 10 — final mental model

Remember this:

```text
                  UNITY CATALOG
                        │
             ┌──────────┴──────────┐
             ▼                     ▼
          Catalog              Governance
             │
           Schema
             │
      ┌──────┼──────┐
      ▼      ▼      ▼
   Tables  Views  Volumes
      │
    Delta
      │
 ┌────┼─────┐
 ▼    ▼     ▼
Bronze Silver Gold
```

And around it:

```text
Auto Loader → ingestion
Jobs        → orchestration
Delta       → transactional storage
Unity Cat.  → governance
Data Quality→ correctness
```

### The one sentence to remember

> **Unity Catalog is the governance layer that organizes, secures and provides a consistent namespace for Databricks data assets.**

This is the foundation you need before moving to **Blog 11 — Jobs & Pipelines** and later **Blog 12 — Auto Loader + Structured Streaming**.